# 033 — Algoritmos genéticos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** `Σf = 9`. Probabilidades: A `3/9≈0.333`, B `2/9≈0.222`, C `3/9≈0.333`, D `1/9≈0.111`. Acumuladas: A hasta 0.333, B hasta 0.556, C hasta 0.889, D hasta 1. Tiradas `[0.10, 0.45, 0.70, 0.95]` → padres **A, B, C, D**. (Nótese que hasta el peor puede reproducirse: la ruleta es sesgo, no determinismo.)

**E2.** Par A×B en punto 2: `10|110` + `00|101` → hijos `10101 (f=3)` y `00110 (f=2)`. Par C×D: `11|100` + `00|010` → `11010 (f=3)` y `00100 (f=1)`. Mejor hijo: 3 (no mejoró el máximo de 3); promedio `9/4 = 2.25` (igual que antes). Una generación sin mutación ni presión fuerte puede no avanzar: el progreso del GA es estadístico, no monótono.

**E3.** Mejor hijo `10101`, bit 4 (`0`) → `10111 (f=4)`: **sube**. Cerca del óptimo la mutación es la única fuente de los bits que faltan si el cruce ya no los puede combinar (nadie en la población tenía el patrón completo).

**E4.** Con elitismo, el mejor de la generación t está garantizado en la generación t+1, así que `max_fitness(t+1) ≥ max_fitness(t)`: la secuencia de máximos es no decreciente — monotonía que el GA puro no tiene, como mostró E2.


In [ ]:
result = run_lab("optimization", seed=33)
assert result["kind"] == "optimization"
assert result["evidence"]
show(result)


In [ ]:
pob = {"A": "10110", "B": "00101", "C": "11100", "D": "00010"}
f = lambda s: s.count("1")
total = sum(f(s) for s in pob.values())
print("E1 probs:", {k: round(f(s)/total, 3) for k, s in pob.items()})

def cruce(p1, p2, punto=2):
    return p1[:punto]+p2[punto:], p2[:punto]+p1[punto:]

h1, h2 = cruce(pob["A"], pob["B"]); h3, h4 = cruce(pob["C"], pob["D"])
hijos = [h1, h2, h3, h4]
print("E2 hijos:", [(h, f(h)) for h in hijos], "promedio", sum(map(f, hijos))/4)
mejor = max(hijos, key=f)
mutado = mejor[:3] + ("1" if mejor[3] == "0" else "0") + mejor[4:]
print(f"E3: {mejor}(f={f(mejor)}) -> {mutado}(f={f(mutado)})")


## Reflexión

1. En la salida del laboratorio, ¿qué evidencia distingue "el GA convergió" de "el GA se estancó en un óptimo local"? ¿Puede distinguirse con una sola corrida?
2. ¿Por qué el cruce de un punto puede producir hijos mejores que ambos padres en OneMax pero rara vez en un problema donde los bits interactúan (epistasis)?
3. Si la mutación fuera `p_m = 0.5` por bit, ¿en qué se convierte el algoritmo? ¿Y con `p_m = 0`?
